# Libraries import 

In [1]:
# Core libraries
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
from pathlib import Path
from typing import Optional, Literal
from typing import Dict, Tuple, List
# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.sparse as sp
import subprocess
# Integration methods
import harmonypy as hm  # Harmony integration
# Quality control
import scrublet  # Doublet detection
import sys
import io
import logging
# Utilities 
import warnings
warnings.filterwarnings('ignore')

# Set display options
sc.settings.verbosity = 3  # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.settings.set_figure_params(dpi=100, facecolor='white')
import torch
torch.set_float32_matmul_precision("high")


/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


# Dataset01: GSE149614

In [3]:
# read data (normalized log1p data + full gene panel)
post_norm_dir = Path("/data1/esraa/Thesis-Project/Data/Processed_data/post_norm_log1p")
adata1 = sc.read_h5ad(post_norm_dir / "adata_149614.h5ad")

In [5]:
print(adata1.shape)  # Check the shape of the data

(66668, 25712)


In [5]:
import scanpy as sc

# adata = sc.read_h5ad("PATH/TO/METRICS_INPUT.h5ad")

# keep only a few obs columns (edit names to match yours)
keep_obs = [c for c in ["pseudotime", "group", "cell_type", "leiden"] if c in adata.obs.columns]

# keep a small gene subset (add your marker genes list here)
marker_genes = ["LYZ","S100A8","S100A9","FCGR3A","MARCO","TREM2","APOE","VEGFA","MKI67"]
keep_genes = [g for g in marker_genes if g in adata.var_names]

mini = adata[:, keep_genes].copy()
mini.obs = mini.obs[keep_obs]
mini.write("mini_metrics_check.h5ad")
print("wrote mini_metrics_check.h5ad with shape", mini.shape)

wrote mini_metrics_check.h5ad with shape (13496, 7)


In [ ]:
# check the column names of the data
print(adata1.obs.columns['patient'])

Index(['sample', 'res.3', 'site', 'patient', 'stage', 'virus', 'celltype',
       'dataset', 'cluster_annotation', 'NL', 'PT', 'PVTT', 'MLN',
       'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt',
       'total_counts_ribo', 'pct_counts_ribo', 'n_genes', 'n_counts',
       'donor_id', 'donor_resolution', 'technology', 'platform', 'sequencer',
       'source_repository', 'compartment', 'tissue', 'tumor_status',
       'disease_status', 'cancer_type', 'disease_group', 'cell_type_raw',
       'celltype_original', 'celltype_l0', 'celltype_l1', 'celltype_l2',
       'is_unclassified', 'is_cycling', 'valid_for_benchmark'],
      dtype='object')


In [ ]:
import pandas as pd

cluster_key = "cluster_annotation"
patient_key = "patient"

# Contingency table
ct = pd.crosstab( 
    adata1.obs[cluster_key],
    adata1.obs[patient_key]
)

# Raw counts
print("Raw cell counts per cluster per patient:")
display(ct)

# Row-normalized (proportion per cluster)
ct_prop = ct.div(ct.sum(axis=1), axis=0)

print("\nProportion of each patient within each cluster:")
display(ct_prop.round(3))

Raw cell counts per cluster per patient:


patient,HCC01,HCC02,HCC03,HCC04,HCC05,HCC06,HCC07,HCC08,HCC09,HCC10
cluster_annotation,,,,,,,,,,
ACTA2+ fibroblast,164,0,16,59,163,475,25,44,61,52
CD1C+ conventional dendritic cell,96,52,269,358,136,333,306,91,204,203
COL1A1+ fibroblast,18,13,15,97,27,176,79,457,148,106
FABP4+ endothelial,382,0,96,163,501,597,67,72,130,103
FCN3+ endothelial,16,0,112,309,336,137,111,71,32,41
IRF4+ plasmacytoid dendritic cell,1,0,12,5,9,17,24,15,21,11
MARCO+ macrophage,39,4,465,206,510,605,377,85,35,99
MMP9+ macrophage,19,16,34,65,4,38,213,635,57,354
SPARC+ endothelial,15,14,27,30,1,13,16,81,33,30



Proportion of each patient within each cluster:


patient,HCC01,HCC02,HCC03,HCC04,HCC05,HCC06,HCC07,HCC08,HCC09,HCC10
cluster_annotation,,,,,,,,,,
ACTA2+ fibroblast,0.155,0.000,0.015,0.056,0.154,0.449,0.024,0.042,0.058,0.049
CD1C+ conventional dendritic cell,0.047,0.025,0.131,0.175,0.066,0.163,0.149,0.044,0.100,0.099
COL1A1+ fibroblast,0.016,0.011,0.013,0.085,0.024,0.155,0.070,0.402,0.130,0.093
FABP4+ endothelial,0.181,0.000,0.045,0.077,0.237,0.283,0.032,0.034,0.062,0.049
FCN3+ endothelial,0.014,0.000,0.096,0.265,0.288,0.118,0.095,0.061,0.027,0.035
IRF4+ plasmacytoid dendritic cell,0.009,0.000,0.104,0.043,0.078,0.148,0.209,0.130,0.183,0.096
MARCO+ macrophage,0.016,0.002,0.192,0.085,0.210,0.249,0.155,0.035,0.014,0.041
MMP9+ macrophage,0.013,0.011,0.024,0.045,0.003,0.026,0.148,0.443,0.040,0.247
SPARC+ endothelial,0.058,0.054,0.104,0.115,0.004,0.050,0.062,0.312,0.127,0.115


In [4]:
print(adata1.obs["cluster_annotation"].value_counts())

cluster_annotation
pro-metastatic hepatocyte            9195
natural killer cell                  6303
pro-tumorigenic hepatocyte           6020
cytotoxic T lymphocytes              3703
mucosal-associated invariant T       3282
monocyte-derived macrophage          3137
plasma B cell                        2762
patient-specific macropahge          2496
MARCO+ macrophage                    2425
TREM2+ macrophage                    2170
FABP4+ endothelial                   2111
regulatory T                         2078
CD1C+ conventional dendritic cell    2048
intermediate T                       2016
effector memory T                    2001
tissue-resident memory T             1878
naïve T                              1616
VEGFA+ macrophage                    1535
MMP9+ macrophage                     1435
FCN3+ endothelial                    1165
COL1A1+ fibroblast                   1136
non-malignant hepatocyte             1113
ACTA2+ fibroblast                    1059
central memory 

In [ ]:
sc.pp.scale(adata1)  # rescale to 0 mean and variance of 1
_ = plt.hist(np.sum(adata1.X, axis=0), bins=100)
plt.xlabel("Mean of transcript totals")
plt.ylabel("Counts")
_ = plt.title("Dynamic range of transcripts in the data")

In [ ]:
sc.tl.pca(adata1, svd_solver="arpack")  # since neighbors() uses PCA values, compute them explicitly here
sc.pp.neighbors(adata1, n_neighbors=4, n_pcs=20)  # compute the kNN distances

In [ ]:
sc.tl.draw_graph(adata1)  # compute the graph
sc.pl.draw_graph(adata1, color="paul15_clusters")  # display the graph

In [ ]:
sc.tl.leiden(adata1, resolution=1, flavor="igraph", n_iterations=2)  # leiden clustering
sc.pl.draw_graph(adata1, color=["leiden", "paul15_clusters"])  # display the graph

In [ ]:
sc.tl.paga(adata1, groups="leiden")
sc.pl.paga(adata1, color=["leiden", "paul15_clusters"])

In [ ]:
sc.tl.draw_graph(adata1, init_pos="paga")
sc.pl.draw_graph(adata1, color=["leiden", "paul15_clusters"], legend_loc="on data")
sc.pl.draw_graph(adata1, color=["Hba-a2", "Elane"])|

# Dataset02: GSE151530

In [13]:
# read data (normalized log1p data + full gene panel)
post_norm_dir = Path("/data1/esraa/Thesis-Project/Data/Processed_data/post_norm_log1p")
adata2 = sc.read_h5ad(post_norm_dir / "adata_151530.h5ad")

In [14]:
# print the columns and the shape of the data
print(adata2.obs.columns)  # Check the observation columns of the data
print(adata2.shape)  # Check the shape of the data

Index(['dataset', 'S_ID', 'Sample', 'Type', 'n_genes_by_counts',
       'total_counts', 'total_counts_mt', 'pct_counts_mt', 'total_counts_ribo',
       'pct_counts_ribo', 'n_genes', 'n_counts', 'log1p_n_genes_by_counts',
       'log1p_total_counts', 'pct_counts_in_top_50_genes',
       'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes',
       'pct_counts_in_top_500_genes', 'log1p_total_counts_mt', 'donor_id',
       'donor_resolution', 'technology', 'platform', 'sequencer',
       'source_repository', 'tissue', 'compartment', 'tumor_status',
       'disease_status', 'cancer_type', 'disease_group', 'cell_type_raw',
       'celltype_original', 'celltype_l0', 'celltype_l1', 'celltype_l2',
       'is_unclassified', 'is_cycling', 'valid_for_benchmark'],
      dtype='object')
(48360, 18667)


In [15]:
#print the unique cell types from column Type
print(adata2.obs["Type"].unique())

['T cells', 'B cells', 'TAMs', 'unclassified', 'Malignant cells', 'TECs', 'CAFs']
Categories (7, object): ['B cells', 'CAFs', 'Malignant cells', 'T cells', 'TAMs', 'TECs', 'unclassified']


# Dataset03: GSE115469

In [ ]:
# read data (normalized log1p data + full gene panel)
post_norm_dir = Path("/data1/esraa/Thesis-Project/Data/Processed_data/post_norm_log1p")
adata3 = sc.read_h5ad(post_norm_dir / "adata_115469.h5ad")

In [ ]:
# print the columns and the shape of the data
print(adata3.obs.columns)  # Check the observation columns of the data
print(adata3.shape)  # Check the shape of the data

Index(['dataset', 'Sample', 'Cell#', 'Cluster#', 'CellType',
       'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt',
       'total_counts_ribo', 'pct_counts_ribo', 'n_genes', 'n_counts',
       'log1p_n_genes_by_counts', 'log1p_total_counts',
       'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes',
       'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes',
       'log1p_total_counts_mt', 'donor_id', 'donor_resolution', 'technology',
       'platform', 'sequencer', 'source_repository', 'tissue', 'compartment',
       'tumor_status', 'disease_status', 'disease_group', 'cancer_type',
       'cell_type_raw', 'celltype_original', 'celltype_l0', 'celltype_l1',
       'celltype_l2', 'is_unclassified', 'is_cycling', 'valid_for_benchmark'],
      dtype='object')
(8346, 20007)


In [ ]:
#print the unique cell types from column Type
print(adata3.obs["CellType"].unique())

['Central_venous_LSECs', 'Cholangiocytes', 'Non-inflammatory_Macrophage', 'alpha-beta_T_Cells', 'Inflammatory_Macrophage', ..., 'Erythroid_Cells', 'Hepatocyte_2', 'Hepatocyte_3', 'Hepatocyte_1', 'Hepatocyte_4']
Length: 20
Categories (20, object): ['Central_venous_LSECs', 'Cholangiocytes', 'Erythroid_Cells', 'Hepatic_Stellate_Cells', ..., 'Portal_endothelial_Cells', 'alpha-beta_T_Cells', 'gamma-delta_T_Cells_1', 'gamma-delta_T_Cells_2']


# Dataset04: GSE125449

In [3]:
# read data (normalized log1p data + full gene panel)
post_norm_dir = Path("/data1/esraa/Thesis-Project/Data/Processed_data/post_norm_log1p")
adata4 = sc.read_h5ad(post_norm_dir / "adata_125449.h5ad")

# print the columns and the shape of the data
print(adata4.obs.columns)  # Check the observation columns of the data
print(adata4.shape)  # Check the shape of the data

Index(['Sample', 'Type', 'batch', 'dataset_concat', 'dataset',
       'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt',
       'total_counts_ribo', 'pct_counts_ribo', 'n_genes', 'n_counts',
       'donor_id', 'donor_resolution', 'technology', 'platform', 'sequencer',
       'source_repository', 'tissue', 'compartment', 'tumor_status',
       'disease_status', 'disease_group', 'cancer_type', 'cell_type_raw',
       'celltype_original', 'celltype_l0', 'celltype_l1', 'celltype_l2',
       'is_unclassified', 'is_cycling', 'valid_for_benchmark'],
      dtype='object')
(8263, 21324)


In [4]:
#print the unique cell types from column Type
print(adata4.obs["Type"].unique())

['CAF', 'TEC', 'T cell', 'B cell', 'TAM', 'unclassified', 'Malignant cell', 'HPC-like']
Categories (8, object): ['B cell', 'CAF', 'HPC-like', 'Malignant cell', 'T cell', 'TAM', 'TEC', 'unclassified']


# Dataset05: GSE140228

In [2]:
# read data (normalized log1p data + full gene panel)
post_norm_dir = Path("/data1/esraa/Thesis-Project/Data/Processed_data/post_norm_log1p")
adata5 = sc.read_h5ad(post_norm_dir / "adata_140228.h5ad")

# print the columns and the shape of the data
print(adata5.obs.columns)  # Check the observation columns of the data
print(adata5.shape)  # Check the shape of the data

Index(['Donor', 'Tissue', 'celltype_sub', 'Platform', 'celltype_global',
       'Sample', 'Histology', 'Tissue_sub', 'dataset', 'technology',
       'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt',
       'total_counts_ribo', 'pct_counts_ribo', 'log1p_n_genes_by_counts',
       'log1p_total_counts', 'pct_counts_in_top_50_genes',
       'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes',
       'pct_counts_in_top_500_genes', 'log1p_total_counts_mt', 'donor_id',
       'donor_resolution', 'platform', 'sequencer', 'source_repository',
       'compartment', 'tissue', 'tumor_status', 'disease_status',
       'cancer_type', 'disease_group', 'cell_type_raw', 'celltype_original',
       'celltype_l0', 'celltype_l1', 'celltype_l2', 'is_unclassified',
       'is_cycling', 'valid_for_benchmark'],
      dtype='object')
(64441, 54574)


In [8]:
# Print unique values as a pandas DataFrame
unique_celltypes = (
    pd.Series(adata5.obs["celltype_sub"].dropna().unique(), name="celltype_sub")
    .sort_values()
    .reset_index(drop=True)
)

# Print number of cells for each subtype
subtype_counts = (
    adata5.obs["celltype_sub"]
    .dropna()
    .value_counts()
    .rename_axis("celltype_sub")
    .reset_index(name="n_cells")
)

display(subtype_counts)

,celltype_sub,n_cells
0,CD8-C6-GZMK,8219
1,CD4-C5-TCF7,5283
2,CD8-C4-CX3CR1,3928
3,CD8-C7-KLRD1,3747
4,Lymphoid-B,3542
5,CD4/CD8-C1-CCR7,3418
6,CD8-C9-SLC4A10,3110
7,CD4-C4-IL7R,2737
8,CD4-C3-ANXA1,2674
9,NK-C7-CD160,2370


## construct helper gene markers for the metrics evaluation 